---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — include notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 📉 Principal Components Regression (PCR)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When predictors are highly correlated, OLS becomes unstable. PCR solves this by first compressing X into a small set of uncorrelated components, then regressing Y on those components instead.

### What you'll learn
- Why multicollinearity breaks OLS and how PCR fixes it
- How PCA components are constructed and what they represent
- Choosing the number of components M via cross-validation
- When PCR beats OLS, Ridge, and Lasso — and when it doesn't

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Hitters dataset (ISLP Chapter 6) ────────────────────────────────────
try:
    hitters = pd.read_csv('https://www.statlearning.com/s/Hitters.csv')
    print(f'✓ Hitters.csv (ISLP): {hitters.shape}')
except Exception:
    hitters = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Hitters.csv')
    print(f'✓ Hitters.csv (fallback): {hitters.shape}')

hitters = hitters.dropna()
hitters_enc = pd.get_dummies(hitters, columns=['League','Division','NewLeague'],
                              drop_first=True, dtype=float)
hitters_enc['log_salary'] = np.log(hitters_enc['Salary'])
hitters_enc = hitters_enc.drop(columns=['Salary'])

target   = 'log_salary'
features = [c for c in hitters_enc.columns if c != target]
X = hitters_enc[features].values.astype(float)
y = hitters_enc[target].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale X — PCA requires standardised features (equal variance)
scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)

print(f'  Features ({len(features)}): {features}')
print(f'  Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'  Python {sys.version.split()[0]} | sklearn {__import__("sklearn").__version__}')
print('✓ Setup complete')


## 📐 Part 1 — The Multicollinearity Problem

OLS finds β by solving **(XᵀX)β = Xᵀy**. When predictors are correlated, **XᵀX** becomes near-singular — small data changes cause wildly different coefficient estimates. Variance explodes.

**PCR approach:**
1. Standardise X
2. Compute PCA: find M orthogonal directions (components) that capture the most variance in X
3. Project X onto those M components → Z (shape n × M, M << p)
4. Run OLS on Z instead of X

Because the components are **uncorrelated by construction**, ZᵀZ is well-conditioned regardless of correlations in X.

In [ ]:
# ── Demonstrate multicollinearity in Hitters ─────────────────────────────────
corr = pd.DataFrame(X_tr_s, columns=features).corr()

# Find the most correlated pairs
pairs = []
for i in range(len(features)):
    for j in range(i+1, len(features)):
        pairs.append((features[i], features[j], abs(corr.iloc[i,j])))
pairs.sort(key=lambda x: -x[2])

print("Top 10 most correlated feature pairs:")
print(f"{'Feature A':15s}  {'Feature B':15s}  {'|r|':>6}")
print("-" * 42)
for a, b, r in pairs[:10]:
    print(f"{a:15s}  {b:15s}  {r:>6.3f}")

# Show condition number — measures how ill-conditioned XᵀX is
XtX = X_tr_s.T @ X_tr_s
cond = np.linalg.cond(XtX)
print(f"\nCondition number of XᵀX: {cond:,.1f}")
print(f"  Rule of thumb: > 1,000 = moderate collinearity, > 10,000 = severe")
print(f"  Here: {'severe' if cond > 10000 else 'moderate' if cond > 1000 else 'mild'} multicollinearity")

# Visualise correlation heatmap
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(features))); ax.set_xticklabels(features, rotation=90, fontsize=8)
ax.set_yticks(range(len(features))); ax.set_yticklabels(features, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Hitters — Feature Correlation Matrix\n'
             'High correlation → multicollinearity → unstable OLS coefficients')
plt.tight_layout()
plt.show()


## 🔬 Part 2 — PCA Step: Explained Variance

Before regressing, we run PCA to find the M principal components. The key question: **how many components M do we need?**

- M = p (all components) → identical to OLS
- M = 1 → single dimension capturing most variance in X
- Optimal M → chosen by cross-validation

**Important:** PCA components are ordered by variance in X, not by how well they predict Y. A component that explains little variance in X might still be important for predicting Y. This is PCR's main weakness — PLS addresses it directly.

In [ ]:
# ── PCA on the standardised training data ────────────────────────────────────
pca = PCA().fit(X_tr_s)
p   = len(features)

cumvar = np.cumsum(pca.explained_variance_ratio_)
m90    = np.searchsorted(cumvar, 0.90) + 1   # components for 90% variance
m95    = np.searchsorted(cumvar, 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
axes[0].bar(range(1, p+1), pca.explained_variance_ratio_ * 100,
            color='#1e5fa8', alpha=0.8)
axes[0].plot(range(1, p+1), cumvar * 100, 'o-', color='#e85d20', lw=2, markersize=5)
axes[0].axhline(90, color='grey', lw=1, ls='--', label='90%')
axes[0].axhline(95, color='grey', lw=1, ls=':', label='95%')
axes[0].axvline(m90, color='#1a7a45', lw=1.5, ls='--',
                label=f'M={m90} → 90%')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot — Hitters Features')
axes[0].legend(fontsize=9)

# Cumulative variance
axes[1].fill_between(range(1, p+1), cumvar * 100, alpha=0.2, color='#1e5fa8')
axes[1].plot(range(1, p+1), cumvar * 100, 'o-', color='#1e5fa8', lw=2)
axes[1].axhline(90, color='#e85d20', lw=1.5, ls='--', label='90% threshold')
axes[1].axhline(95, color='#1a7a45', lw=1.5, ls='--', label='95% threshold')
axes[1].set_xlabel('Number of Components M')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend(fontsize=9)

plt.suptitle('PCA on Hitters Predictors', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Components to explain 90% of variance in X: M = {m90}")
print(f"Components to explain 95% of variance in X: M = {m95}")
print(f"Full dimensionality: p = {p}")
print()
print("Component loadings (first 3 components — top contributing features):")
loadings = pd.DataFrame(pca.components_[:3].T, index=features,
                        columns=[f'PC{i+1}' for i in range(3)])
for pc in loadings.columns:
    top = loadings[pc].abs().nlargest(5).index.tolist()
    print(f"  {pc}: {top}")


## 📊 Part 3 — PCR: Choosing M by Cross-Validation

We fit OLS on the first M principal components. M is the only hyperparameter — chosen by CV.

**PCR pipeline:** StandardScaler → PCA(n_components=M) → LinearRegression

In [ ]:
# ── PCR: cross-validate over M = 1 … p ──────────────────────────────────────
M_range  = range(1, p + 1)
cv_mses  = []

for M in M_range:
    pipe = Pipeline([
        ('pca', PCA(n_components=M)),
        ('ols', LinearRegression())
    ])
    mse = -cross_val_score(pipe, X_tr_s, y_tr, cv=10,
                            scoring='neg_mean_squared_error').mean()
    cv_mses.append(mse)

best_M    = list(M_range)[np.argmin(cv_mses)]
best_mse  = min(cv_mses)

# OLS baseline (M = p = all components, identical to full OLS)
ols_mse = -cross_val_score(LinearRegression(), X_tr_s, y_tr, cv=10,
                            scoring='neg_mean_squared_error').mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(list(M_range), cv_mses, 'o-', color='#1e5fa8', lw=2,
        markersize=4, label='PCR 10-fold CV MSE')
ax.axhline(ols_mse, color='grey', lw=1.5, ls='--',
           label=f'Full OLS (M=p={p}) CV MSE = {ols_mse:.4f}')
ax.axvline(best_M, color='#e85d20', lw=2, ls='--',
           label=f'Best M = {best_M}  (CV MSE = {best_mse:.4f})')
ax.set_xlabel('Number of PCA Components M')
ax.set_ylabel('10-fold CV MSE (log salary)')
ax.set_title('PCR: Choosing the Number of Components via Cross-Validation')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Fit best PCR on full training set, evaluate on test set
pcr_best = Pipeline([('pca', PCA(n_components=best_M)),
                     ('ols', LinearRegression())])
pcr_best.fit(X_tr_s, y_tr)
pcr_test_mse = mean_squared_error(y_te, pcr_best.predict(X_te_s))

# OLS on full data
ols_full = LinearRegression().fit(X_tr_s, y_tr)
ols_test_mse = mean_squared_error(y_te, ols_full.predict(X_te_s))

print(f"Best M = {best_M} components (out of p = {p})")
print(f"Variance explained by M={best_M} components: {pca.explained_variance_ratio_[:best_M].sum():.1%}")
print()
print(f"{'Model':25s}  {'CV MSE':>10}  {'Test MSE':>10}")
print("-" * 50)
print(f"{'OLS (all features)':25s}  {ols_mse:>10.4f}  {ols_test_mse:>10.4f}")
print(f"{'PCR (M='+str(best_M)+')':25s}  {best_mse:>10.4f}  {pcr_test_mse:>10.4f}")
print()
print("📌 PCR compresses correlated features into uncorrelated components.")
print(f"   M={best_M} components capture {pca.explained_variance_ratio_[:best_M].sum():.1%} of variance in X.")
print("   When M < p, PCR is a form of dimensionality reduction before regression.")


## ✅ Part 4 — Interpreting PCR Coefficients

PCR returns coefficients in **component space** — each β corresponds to a principal component, not an original feature. To interpret in terms of original features, we back-transform:

```
β_original = V @ β_pcr
```

where V is the p × M loading matrix from PCA.

This gives us the implied coefficient for each original feature, which we can then examine with `sm.OLS` for inference.

In [ ]:
# ── Back-transform PCR coefficients to original feature space ────────────────
pca_fit  = pcr_best.named_steps['pca']
ols_fit  = pcr_best.named_steps['ols']

# β in original (scaled) feature space
beta_orig = pca_fit.components_[:best_M].T @ ols_fit.coef_
intercept = ols_fit.intercept_ - (pca_fit.mean_ @ beta_orig if pca_fit.mean_ is not None else 0)

print(f"PCR implied coefficients in original feature space (M={best_M}):")
print(f"{'Feature':20s}  {'PCR β':>10}  {'OLS β (full)':>13}")
print("-" * 48)
ols_sm   = sm.OLS(y_tr, sm.add_constant(X_tr_s)).fit()
for fname, b_pcr, b_ols in zip(features, beta_orig, ols_sm.params[1:]):
    diff = '←' if abs(b_pcr - b_ols) > 0.05 else ''
    print(f"{fname:20s}  {b_pcr:>10.4f}  {b_ols:>13.4f}  {diff}")

print(f"\nIntercept: PCR={intercept:.4f}  OLS={ols_sm.params[0]:.4f}")
print()
print("📌 PCR and OLS coefficients differ because PCR drops low-variance components.")
print("   Features that load heavily on dropped components will differ most.")
print("   PCR does NOT provide p-values — use Ridge/OLS if inference is needed.")
print()
print("When to use PCR vs alternatives:")
print("  PCR   — high multicollinearity, want dimensionality reduction, p < n")
print("  Ridge — same setting but want to keep all features with shrinkage")
print("  PLS   — same as PCR but components chosen to predict Y, not just explain X")


## 💼 Exercise

**Task 1 — Compare all shrinkage methods:** Using the same Hitters train/test split, compute test MSE for OLS, Ridge (RidgeCV), Lasso (LassoCV), and PCR (best M). Which wins? Fill in the comparison table.

**Task 2 — Component stability:** Refit PCA with M=3 and plot the biplot (PC1 vs PC2 with feature loadings overlaid). Which features point in similar directions? What does that tell you about their correlations?

**Task 3 — How many components for salary?** CV picks the optimal M for minimising prediction error. Does that M match the number needed to explain 90% of variance in X? What does any discrepancy tell you?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables available:
#   X_tr_s, X_te_s, y_tr, y_te, features
#   pcr_best — fitted Pipeline (PCA + OLS)
#   best_M   — optimal number of components

# Task 1 — Compare all methods
ridge = RidgeCV(cv=10).fit(X_tr_s, y_tr)
lasso = LassoCV(cv=10, random_state=42).fit(X_tr_s, y_tr)

results = {
    'OLS':   mean_squared_error(y_te, LinearRegression().fit(X_tr_s,y_tr).predict(X_te_s)),
    'Ridge': mean_squared_error(y_te, ridge.predict(X_te_s)),
    'Lasso': mean_squared_error(y_te, lasso.predict(X_te_s)),
    f'PCR (M={best_M})': pcr_test_mse,
}
print("Test MSE comparison:")
for name, mse in sorted(results.items(), key=lambda x: x[1]):
    print(f"  {name:20s}: {mse:.4f}  (RMSE={mse**0.5:.4f})")

# Task 2 — Biplot (your code here)

# Task 3 — Compare CV-optimal M to 90% variance threshold
print(f"\nCV-optimal M:          {best_M}")
print(f"M for 90% X-variance:  {m90}")
print(f"Difference:            {best_M - m90:+d}")


In [ ]:
# @title 📝 Quiz — Principal Components Regression
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** Why must features be standardised before applying PCR?
# @markdown **Q2:** How are the principal components chosen in PCR — what do they maximise?
# @markdown **Q3:** When M = p in PCR, what model is it equivalent to?
# @markdown **Q4:** What is the main disadvantage of PCR compared to PLS?
# @markdown **Q5:** Name a situation where you would prefer Ridge over PCR.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Principal Components Regression"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*